In [1]:
# load the file wanted
from imu_uwb_pose.config import config
import pytorch_lightning as pl
from imu_uwb_pose.training.imu_uwb_pose_model import imu_uwb_pose_model
from imu_uwb_pose.utils import get_smpl_output
from imu_uwb_pose.training.predict_datamodule import SimplePredictDataModule
from imu_uwb_pose.utils import r6d_to_axis_angle, axis_angle_to_rotation_matrix, rotation_matrix_to_axis_angle
import smplx
import torch
import os

EXPERIMENT = 'bilstm_2_layer_footposer_mocap_1e-4'
FILE_NAME = 'FootPoser_mocap/helen/activities1.pt'

config = config(EXPERIMENT, 'FootPoser', name='helen')

file_loc = f'{config.processed_pose}/{FILE_NAME}'

def split_into_chunks(tensor, chunk_size):
    return [tensor[i:i + chunk_size] for i in range(0, tensor.shape[0], chunk_size)]


# load file and segment it into batches
pose_data = torch.load(file_loc, map_location='cpu', weights_only=True)
x_data = split_into_chunks(pose_data['x'], config.max_sample_length)
y_data = split_into_chunks(pose_data['y'], config.max_sample_length)
joints_data = split_into_chunks(pose_data['joints'], config.max_sample_length)

predict_datamodule = SimplePredictDataModule(
    x_data,
    y_data,
    joints_data,
    batch_size=config.batch_size,
    num_workers=config.num_workers,
)

# load the model
best_model_txt_path = os.path.join(config.checkpoint_path, "best_model.txt")
with open(best_model_txt_path, "r") as f:
    lines = f.readlines()
    best_model_path = lines[0].strip()

model = imu_uwb_pose_model.load_from_checkpoint(
    best_model_path,
    map_location=config.device,
    config=config
)

trainer = pl.Trainer(
    accelerator='gpu' if torch.cuda.is_available() else 'cpu',
    devices=[0] if config.device.type == 'cuda' else 0,
    fast_dev_run=False,
)

outputs = trainer.predict(model, predict_datamodule)

smpl = smplx.create(config.body_model, model_type='smplx',
        gender='neutral', use_face_contour=False,
        batch_size=1,
        ext='npz',
        age='adult').to(config.device)

# get smpl output x and y
pred_output = []
true_output = []
length_output = []
for i in range(len(outputs)):
    # convert pred and true to axis angle
    pred_r6d = outputs[i]['pred'].reshape(-1, 6)
    true_r6d = outputs[i]['true'].reshape(-1, 6)
    lengths = outputs[i]['lengths']

    pred_aa = r6d_to_axis_angle(pred_r6d).reshape(-1, 66).to(config.device)
    true_aa = r6d_to_axis_angle(true_r6d).reshape(-1,66).to(config.device)

    pred_output.append(pred_aa)
    true_output.append(true_aa)
    length_output.extend(lengths)
        
pred_output = torch.concatenate(pred_output)
true_output = torch.concatenate(true_output)
pred_vertices, pred_joints, pred_faces = get_smpl_output(smpl, pred_output, config)
true_vertices,true_joints,true_faces = get_smpl_output(smpl, true_output, config)

print(f'Number of valid frames {sum(length_output)}')


Using default `ModelCheckpoint`. Consider installing `litmodels` package to enable `LitModelCheckpoint` for automatic upload to the Lightning model registry.
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs
/home/lichard/miniconda3/envs/imu_uwb_pose/lib/python3.10/site-packages/pytorch_lightning/trainer/connectors/logger_connector/logger_connector.py:76: Starting from v1.9.0, `tensorboardX` has been removed as a dependency of the `pytorch_lightning` package, due to potential conflicts with other packages in the ML ecosystem. For this reason, `logger=True` will use `CSVLogger` as the default logger, unless the `tensorboard` or `tensorboardX` packages are found. Please `pip install lightning[extra]` or one of them to enable TensorBoard support by default
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [GPU-0bb37768-064b-8608-e924-491e9c107d70]


Predicting: |          | 0/? [00:00<?, ?it/s]

Number of valid frames 6664


In [2]:
FRAME_IDX = 2000

def angle_err_at_frame(pred_angle, true_angle):
    pred_rot_mat = axis_angle_to_rotation_matrix(pred_angle)
    true_rot_mat = axis_angle_to_rotation_matrix(true_angle)

    true_rot_trans = torch.transpose(true_rot_mat, 1,2)
    angles = torch.linalg.norm(rotation_matrix_to_axis_angle(torch.matmul(pred_rot_mat, true_rot_trans)),dim=1)
    return angles

angle_errs = angle_err_at_frame(pred_output[FRAME_IDX].reshape(22,3), true_output[FRAME_IDX].reshape(22,3)).cpu()
print(f'angle err {sum(angle_errs)}')

# convert angle errs to normal distribution
from scipy.stats import norm
import numpy as np

mu = angle_errs.mean()
std = angle_errs.std()
cdf_vals = norm.cdf(angle_errs, loc=mu, scale=std)
cdf_vals = np.clip(cdf_vals, 0.0, 1.0)


angle err 3.5835840702056885


In [ ]:
import numpy as np
import open3d as o3d
import torch
def render_smplx_frame(
    faces,
    frame_vertices,
    output_path=None,
    width=1280,
    height=720,
):
    """
    Render a single SMPL-X frame using Open3D.
    Args:
        faces: Face indices for the SMPL-X mesh (N, 3).
        frame_vertices: (N, 3) vertex array or torch tensor for the desired frame.
        output_path: If provided, renders offscreen and saves the PNG at this path.
        window_name: Title for the Open3D window (used only when `output_path` is None).
        width: Width of the offscreen render (only used when `output_path` is set).
        height: Height of the offscreen render.
    Example:
        smpl_model = smplx.create(...).to(device)
        verts = smpl_model(**smpl_params).vertices[0]
        render_smplx_frame(faces, verts, output_path="frame0.png")
    """
    vertices = frame_vertices.detach().cpu().numpy() if isinstance(frame_vertices, torch.Tensor) else np.asarray(frame_vertices)
    mesh = o3d.geometry.TriangleMesh()
    mesh.triangles = o3d.utility.Vector3iVector(np.asarray(faces, dtype=np.int32))
    mesh.vertices = o3d.utility.Vector3dVector(vertices)
    mesh.compute_vertex_normals()
    if output_path:
        renderer = o3d.visualization.rendering.OffscreenRenderer(width, height)
        renderer.scene.set_background([1, 1, 1, 1])
        material = o3d.visualization.rendering.MaterialRecord()
        material.shader = "defaultLit"
        renderer.scene.add_geometry("mesh", mesh, material)
        bbox = mesh.get_axis_aligned_bounding_box()
        center = bbox.get_center()
        extent = bbox.get_extent().max()
        eye = center + np.array([0, extent * 1.8, 0])
        up = np.array([0, 1, 0])
        renderer.scene.camera.look_at(center, eye, up)
        image = renderer.render_to_image()
        o3d.io.write_image(output_path, image)
        return output_path

import numpy as np
import open3d as o3d

def make_smpl_geoms(joints, errors, cfg):
    joints = np.asarray(joints).reshape(22, 3)
    errors = np.asarray(errors).reshape(22,)

    norm_errors = errors - errors.min()
    max_err = norm_errors.max()
    if max_err > 1e-8:
        norm_errors = norm_errors / max_err

    colors = np.stack([norm_errors, np.zeros_like(norm_errors), 1.0 - norm_errors], axis=1)

    def rotation_from_z(direction):
        base = np.array([0.0, 0.0, 1.0])
        direction = direction / np.linalg.norm(direction)
        dot = np.clip(np.dot(base, direction), -1.0, 1.0)
        if np.isclose(dot, 1.0):
            return np.eye(3)
        if np.isclose(dot, -1.0):
            return np.array([[1.0, 0.0, 0.0], [0.0, -1.0, 0.0], [0.0, 0.0, -1.0]])
        cross = np.cross(base, direction)
        s = np.linalg.norm(cross)
        k = np.array([[0, -cross[2], cross[1]], [cross[2], 0, -cross[0]], [-cross[1], cross[0], 0]])
        return np.eye(3) + k + k @ k * ((1 - dot) / (s ** 2))

    joint_meshes = []
    for joint, color in zip(joints, colors):
        sphere = o3d.geometry.TriangleMesh.create_sphere(radius=0.04)
        sphere.compute_vertex_normals()
        sphere.paint_uniform_color(color.tolist())
        sphere.translate(joint.tolist())
        joint_meshes.append(sphere)

    skeleton = cfg.get_smpl_skeleton().numpy()
    bone_meshes = []
    for start_idx, end_idx in skeleton:
        start = joints[start_idx]
        end = joints[end_idx]
        direction = end - start
        length = np.linalg.norm(direction)
        if length <= 1e-6:
            continue
        cylinder = o3d.geometry.TriangleMesh.create_cylinder(radius=0.01, height=length)
        cylinder.compute_vertex_normals()
        rotation = rotation_from_z(direction)
        cylinder.rotate(rotation, center=(0, 0, 0))
        cylinder.translate(((start + end) / 2.0).tolist())
        cylinder.paint_uniform_color([0, 0, 0])
        bone_meshes.append(cylinder)

    return joint_meshes, bone_meshes

def render_angle_err_skeleton(joints, errors, cfg, out_path, width=1280, height=720):
    joints = np.asarray(joints).reshape(22, 3)
    joint_meshes, bone_meshes = make_smpl_geoms(joints, errors, cfg)

    try:
        renderer = o3d.visualization.rendering.OffscreenRenderer(width, height)
        renderer.scene.set_background([1, 1, 1, 1])

        joint_mat = o3d.visualization.rendering.MaterialRecord()
        joint_mat.shader = "defaultLit"
        bone_mat = o3d.visualization.rendering.MaterialRecord()
        bone_mat.shader = "defaultLit"

        for idx, mesh in enumerate(joint_meshes):
            renderer.scene.add_geometry(f"joint_{idx}", mesh, joint_mat)

        for idx, mesh in enumerate(bone_meshes):
            renderer.scene.add_geometry(f"bone_{idx}", mesh, bone_mat)

        bbox = o3d.geometry.AxisAlignedBoundingBox.create_from_points(
            o3d.utility.Vector3dVector(joints)
        )
        center = bbox.get_center()
        extent = bbox.get_extent().max()

        eye = center + np.array([0, extent * 1.8, 0])
        up = np.array([0, 1, 0])
        renderer.scene.camera.look_at(center, eye, up)

        img = renderer.render_to_image()
        o3d.io.write_image(out_path, img)
        return out_path

    except Exception as e:
        print("Open3D offscreen failed", e)


render_smplx_frame(true_faces, true_vertices[FRAME_IDX], output_path=f"./figs/error_diagrams/true_smpl_{FRAME_IDX}.png")
render_smplx_frame(pred_faces, pred_vertices[FRAME_IDX], output_path=f"./figs/error_diagrams/pred_smpl_{FRAME_IDX}.png")
render_angle_err_skeleton(pred_joints[FRAME_IDX][:22], cdf_vals, config, f'./figs/error_diagrams/pred_angle_errs_{FRAME_IDX}.png')



[Open3D INFO] EGL headless mode enabled.
FEngine (64 bits) created at 0x2e491730 (threading is enabled)
EGL(1.5)
OpenGL(4.1)
[Open3D INFO] EGL headless mode enabled.
FEngine (64 bits) created at 0x2e491730 (threading is enabled)
EGL(1.5)
OpenGL(4.1)
[Open3D INFO] EGL headless mode enabled.
FEngine (64 bits) created at 0x2e491730 (threading is enabled)
EGL(1.5)
OpenGL(4.1)


'./figs/error_diagrams/pred_angle_errs_2000.png'